# RAG Memory: Learning Through Conversation

In the workflow notebook the index was static — a one-time dump of the Vestas report. Here we flip it: the index becomes the agent's **memory**, and the agent can both **search it** and **add to it** as the conversation unfolds.

The agent gets two tools:

- `search_memory(query)` — semantic search over what we already know about the user.
- `store_memory(content)` — save a new fact for later.

Memory persists across invocations because the vector store lives outside any single run.

## Setup

Load environment variables from your `.env` file. The Azure endpoint is pasted directly on the embeddings model below.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI, AzureOpenAIEmbeddings

llm = ChatOpenAI(base_url=os.environ["OPENAI_ENDPOINT"], model="gpt-5.4-mini")
embeddings = AzureOpenAIEmbeddings(
    model="text-embedding-3-large",
    azure_endpoint="",  # TODO: paste your Azure endpoint here
)

print("Setup complete.")

## Step 1: Create and seed the memory store

We'll seed it with a handful of facts about a fake user named Alex so we have something to search.

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document

memory_store = InMemoryVectorStore(embeddings)

seed_memories = [
    Document(page_content="User's name is Alex. They are a software engineer based in Copenhagen."),
    Document(page_content="User prefers Python over JavaScript for backend development."),
    Document(page_content="User is allergic to shellfish and peanuts."),
    Document(page_content="User's favorite framework is FastAPI."),
]
memory_store.add_documents(seed_memories)

print(f"Seeded {len(seed_memories)} memories.")

## Step 2: Define the memory tools

The agent will decide when to call each tool based on the docstring.

In [ ]:
from langchain.tools import tool


@tool
def search_memory(query: str) -> str:
    """Search memory for information about the user.
    Use this when answering anything that might involve known user facts."""
    docs = memory_store.similarity_search(query, k=3)
    if not docs:
        return "No relevant memories found."
    return "\n".join(f"- {doc.page_content}" for doc in docs)


@tool
def store_memory(content: str) -> str:
    """Store a new fact about the user in memory.
    Use this whenever the user shares something worth remembering."""
    memory_store.add_documents([Document(page_content=content)])
    return f"Stored in memory: {content}"

## Step 3: Build the agent

The system prompt nudges the agent to be proactive — search before answering, store anything new.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[search_memory, store_memory],
    system_prompt=(
        "You are a personal assistant with a memory system.\n\n"
        "Guidelines:\n"
        "- When the user tells you something about themselves, store it with `store_memory`.\n"
        "- Before answering a question that might need user info, search with `search_memory`.\n"
        "- Use retrieved memories naturally in your responses."
    ),
)

## Step 4: Recall a pre-seeded fact

The agent should call `search_memory` and find the answer in the seeded data.

In [ ]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "What do you know about my programming preferences?"}]
})

for message in result["messages"]:
    message.pretty_print()

## Step 5: Teach it something new

The agent should call `store_memory` to save the new fact.

In [ ]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "I just started using PostgreSQL for my new side project."}]
})

for message in result["messages"]:
    message.pretty_print()

## Step 6: Recall the new fact in a fresh conversation

Notice we send a brand-new `messages` list (no history). The agent only knows what's in the **memory store** — and the fact we just saved is there.

In [ ]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "Which database am I using for my side project?"}]
})

for message in result["messages"]:
    message.pretty_print()

## Try it yourself 🛠️

Add a third tool, `forget_memory`, that lets the user delete a fact.

For simplicity, use a *placeholder* implementation — `InMemoryVectorStore` doesn't support proper deletion out of the box, so just have the tool return a confirmation string. The point is to see the agent **decide** to call it.

1. Define a `forget_memory(content: str)` tool with a clear docstring (e.g. *"Forget a previously stored fact about the user."*).
2. Add it to the agent's `tools` list.
3. Update the system prompt so the agent knows when to use it.
4. Test by saying something like *"Forget that I'm allergic to peanuts."* — does the agent call `forget_memory`?

In [ ]:
# Your solution here

@tool
def forget_memory(content: str) -> str:
    """..."""  # TODO: write a docstring describing when to call this tool.
    # TODO: return a confirmation string (real deletion is out of scope here).
    ...


agent = create_agent(
    model=llm,
    tools=[search_memory, store_memory, forget_memory],  # forget_memory added
    system_prompt=(
        # TODO: extend the prompt so the agent knows about forget_memory.
        "You are a personal assistant with a memory system."
    ),
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Please forget that I'm allergic to peanuts."}]
})

for message in result["messages"]:
    message.pretty_print()